# 01 — Data Exploration

Explore the 15 benchmark datasets used in this study: summary statistics, feature types, missing values, class balance, correlations, and scaling characteristics.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.registry import load_dataset, get_all_dataset_names
from src.utils.logging import setup_logging

setup_logging()

# --- Publication-quality defaults ---
sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

SAVE_DPI = 300

# Consistent color palette for task types and model families
TASK_COLORS = {'binary': '#4C72B0', 'multiclass': '#DD8452', 'regression': '#55A868'}
FAMILY_COLORS = {'GBDT': '#4C72B0', 'DL': '#DD8452', 'Foundation': '#55A868'}

os.makedirs('../results/figures', exist_ok=True)

%matplotlib inline

In [ ]:
# Load all datasets and collect summary statistics
dataset_names = get_all_dataset_names()
summaries = []
datasets_cache = {}  # cache for reuse in later cells

for name in dataset_names:
    try:
        X, y, info = load_dataset(name)
        datasets_cache[name] = (X, y, info)
        summary = {
            'Dataset': name,
            'Samples': len(X),
            'Features': X.shape[1],
            'Numerical': len(info.num_columns),
            'Categorical': len(info.cat_columns),
            'Task': info.task_type,
            'Classes': info.n_classes or '-',
            'Missing (%)': round(X.isnull().sum().sum() / X.size * 100, 2),
        }
        summaries.append(summary)
    except Exception as e:
        print(f'Could not load {name}: {e}')

summary_df = pd.DataFrame(summaries)
summary_df.style.set_caption('Table 1: Summary of Benchmark Datasets')

In [ ]:
# Dataset sizes bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

summary_df.sort_values('Samples').plot.barh(
    x='Dataset', y='Samples', ax=axes[0], legend=False, color='steelblue')
axes[0].set_title('Number of Samples')
axes[0].set_xlabel('Samples')

summary_df.sort_values('Features').plot.barh(
    x='Dataset', y='Features', ax=axes[1], legend=False, color='coral')
axes[1].set_title('Number of Features')
axes[1].set_xlabel('Features')

plt.tight_layout()
plt.savefig('../results/figures/dataset_overview.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# Target distribution for each dataset
n_datasets = len(datasets_cache)
n_cols = 5
n_rows = int(np.ceil(n_datasets / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = axes.flat

for idx, name in enumerate(datasets_cache):
    X, y, info = datasets_cache[name]
    ax = axes[idx]
    if info.task_type in ('binary', 'multiclass'):
        pd.Series(y).value_counts().sort_index().plot.bar(
            ax=ax, color=TASK_COLORS.get(info.task_type, 'steelblue'))
        ax.set_title(f'{name}\n({info.task_type}, classes={info.n_classes})')
    else:
        ax.hist(y, bins=50, color=TASK_COLORS['regression'], edgecolor='white')
        ax.set_title(f'{name}\n(regression)')
    ax.set_xlabel('')

# Hide unused subplot axes
for idx in range(len(datasets_cache), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Target Distributions', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/target_distributions.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# Feature correlation heatmaps — one representative per task type
representative_datasets = {
    'Binary classification': 'adult',
    'Multiclass classification': 'jannis',
    'Regression': 'wine_quality',
}

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

for ax, (task_label, ds_name) in zip(axes, representative_datasets.items()):
    X, y, info = datasets_cache[ds_name]
    # Use only numerical columns for correlation
    num_cols = info.num_columns
    if len(num_cols) > 15:
        # Show top-15 by variance for readability
        variances = X[num_cols].var().sort_values(ascending=False)
        num_cols = variances.head(15).index.tolist()

    corr = X[num_cols].corr()
    sns.heatmap(corr, annot=len(num_cols) <= 12, fmt='.2f', cmap='coolwarm',
                center=0, square=True, ax=ax, cbar_kws={'shrink': 0.8})
    ax.set_title(f'{ds_name}\n({task_label})')

plt.suptitle('Feature Correlations (Representative Datasets)', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/correlation_heatmaps.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

## Feature Type Breakdown

Number of numerical vs. categorical features per dataset, and overall feature-type composition.

In [ ]:
# Feature type breakdown: stacked bar chart
feature_df = summary_df[['Dataset', 'Numerical', 'Categorical', 'Task']].copy()
feature_df = feature_df.set_index('Dataset')

fig, ax = plt.subplots(figsize=(12, 6))
feature_df[['Numerical', 'Categorical']].plot.barh(
    stacked=True, ax=ax, color=['steelblue', 'coral'])
ax.set_xlabel('Number of Features')
ax.set_title('Feature Type Breakdown per Dataset')
ax.legend(title='Feature Type')
plt.tight_layout()
plt.savefig('../results/figures/feature_type_breakdown.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

# Table view
print("Feature type composition:")
feature_df['Pct Categorical'] = (
    feature_df['Categorical'] / (feature_df['Numerical'] + feature_df['Categorical']) * 100
).round(1)
feature_df

## Missing Value Analysis

Per-dataset missing value summary. Understanding missingness patterns is important because different model families handle missing data differently: GBDTs handle NaN natively, while deep learning models require imputation.

In [ ]:
# Missing value analysis across all datasets
missing_rows = []

for name, (X, y, info) in datasets_cache.items():
    total_cells = X.size
    total_missing = X.isnull().sum().sum()
    features_with_missing = (X.isnull().sum() > 0).sum()

    missing_rows.append({
        'Dataset': name,
        'Total Cells': total_cells,
        'Missing Cells': total_missing,
        'Missing (%)': round(total_missing / total_cells * 100, 2) if total_cells > 0 else 0,
        'Features with Missing': features_with_missing,
        'Features Total': X.shape[1],
    })

missing_df = pd.DataFrame(missing_rows)

# Highlight datasets that have missing values
has_missing = missing_df[missing_df['Missing Cells'] > 0]
if has_missing.empty:
    print("None of the 15 datasets contain missing values after download.")
    print("This is a limitation: the benchmark does not test imputation strategies.")
else:
    print(f"{len(has_missing)} dataset(s) contain missing values:")

missing_df.style.set_caption('Table 2: Missing Value Summary').bar(
    subset=['Missing (%)'], color='#FFCCCB')

## Class Imbalance Analysis

For classification datasets, the ratio between the majority and minority class can significantly affect model performance. We report the imbalance ratio (majority count / minority count) and per-class proportions.

In [ ]:
# Class imbalance ratios for classification datasets
imbalance_rows = []

for name, (X, y, info) in datasets_cache.items():
    if info.task_type not in ('binary', 'multiclass'):
        continue
    counts = pd.Series(y).value_counts()
    majority = counts.iloc[0]
    minority = counts.iloc[-1]
    imbalance_ratio = round(majority / minority, 2)
    majority_pct = round(majority / len(y) * 100, 1)
    minority_pct = round(minority / len(y) * 100, 1)
    imbalance_rows.append({
        'Dataset': name,
        'Task': info.task_type,
        'Classes': info.n_classes,
        'Majority Count': majority,
        'Minority Count': minority,
        'Imbalance Ratio': imbalance_ratio,
        'Majority (%)': majority_pct,
        'Minority (%)': minority_pct,
    })

imbalance_df = pd.DataFrame(imbalance_rows)

# Bar chart of imbalance ratios
fig, ax = plt.subplots(figsize=(10, 5))
colors = [TASK_COLORS.get(t, 'steelblue') for t in imbalance_df['Task']]
imbalance_df.sort_values('Imbalance Ratio').plot.barh(
    x='Dataset', y='Imbalance Ratio', ax=ax, legend=False, color=colors)
ax.axvline(x=1.0, color='gray', linestyle='--', linewidth=0.8, label='Balanced (1:1)')
ax.set_xlabel('Imbalance Ratio (majority / minority)')
ax.set_title('Class Imbalance Ratios (Classification Datasets)')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/class_imbalance.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

imbalance_df.style.set_caption('Table 3: Class Imbalance Summary')

## Dataset Size vs. Dimensionality

A scatter plot showing each dataset's position in the samples-vs-features space, colored by task type. This helps visualize the diversity of the benchmark suite and identify which regimes (low-dimensional, high-dimensional, small, large) are covered.

In [ ]:
# Dataset size vs. dimensionality scatter plot
fig, ax = plt.subplots(figsize=(10, 7))

for task_type, color in TASK_COLORS.items():
    mask = summary_df['Task'] == task_type
    subset = summary_df[mask]
    ax.scatter(subset['Features'], subset['Samples'],
               c=color, s=120, alpha=0.8, edgecolors='white', linewidth=0.8,
               label=task_type.replace('_', ' ').title(), zorder=3)

# Annotate each point with dataset name
for _, row in summary_df.iterrows():
    ax.annotate(row['Dataset'], (row['Features'], row['Samples']),
                textcoords='offset points', xytext=(6, 4), fontsize=9, alpha=0.85)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of Features (log scale)')
ax.set_ylabel('Number of Samples (log scale)')
ax.set_title('Dataset Landscape: Samples vs. Features')
ax.legend(title='Task Type')
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/dataset_landscape.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()